# Baseline DVFM Experiment: Latent vs No-Latent vs CoxPH

This notebook visualizes the revised experiment in which all models receive only baseline covariates \(x\) at test time.

Changes in this version:

1. Event-time prediction is summarized by the first interpolated survival-curve crossing \(S(t)=0.5\).
2. The reported time error is **median-time MAE**.
3. Weibull shape and scale are bounded for both neural models.
4. A plain lifelines CoxPH baseline is included.
5. The fraction of subjects whose predicted survival curve reaches 0.5 within the training-defined grid is reported explicitly.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("outputs/support_clayton_frailty_tau05_baseline_latent_vs_no_latent")

results = json.loads((OUTPUT_DIR / "results.json").read_text(encoding="utf-8"))
latent_history = pd.read_csv(OUTPUT_DIR / "latent_training_history.csv")
no_latent_history = pd.read_csv(OUTPUT_DIR / "no_latent_training_history.csv")
predictions = pd.read_csv(OUTPUT_DIR / "baseline_model_test_predictions.csv")
brier = pd.read_csv(OUTPUT_DIR / "baseline_oracle_brier_curves.csv")

comparison = results["baseline_survival_comparison"]
primary = comparison["all_test_primary"]

print("Estimand:", comparison["estimand"])
print("Median definition:", comparison["median_definition"])
print("Weibull constraints:", results["weibull_constraints_normalized_time"])


## Headline metrics

In [ ]:
summary = pd.DataFrame({
    "Model": ["Latent DVFM", "No-latent", "CoxPH"],
    "Oracle CI": [
        primary["latent_oracle_ci"],
        primary["no_latent_oracle_ci"],
        primary["coxph_oracle_ci"],
    ],
    "Oracle IBS": [
        primary["latent_oracle_ibs"],
        primary["no_latent_oracle_ibs"],
        primary["coxph_oracle_ibs"],
    ],
    "Median-time MAE": [
        primary["latent_median_time_mae"],
        primary["no_latent_median_time_mae"],
        primary["coxph_median_time_mae"],
    ],
    "Median crossing fraction": [
        primary["latent_median_crossing_fraction"],
        primary["no_latent_median_crossing_fraction"],
        primary["coxph_median_crossing_fraction"],
    ],
})
summary


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
metrics = [
    ("Oracle CI", True),
    ("Oracle IBS", False),
    ("Median-time MAE", False),
    ("Median crossing fraction", True),
]
for ax, (metric, higher_better) in zip(axes, metrics):
    ax.bar(summary["Model"], summary[metric])
    ax.set_title(metric + (" ↑" if higher_better else " ↓"))
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## Training dynamics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    latent_history["epoch"],
    latent_history["validation_reconstruction_nll"],
    label="Latent validation NLL",
)
ax.plot(
    no_latent_history["epoch"],
    no_latent_history["validation_reconstruction_nll"],
    label="No-latent validation NLL",
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation reconstruction NLL")
ax.set_title("Within-model checkpoint criterion")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## Oracle Brier score over time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(brier["time"], brier["latent_oracle_brier"], label="Latent DVFM")
ax.plot(brier["time"], brier["no_latent_oracle_brier"], label="No-latent")
ax.plot(brier["time"], brier["coxph_oracle_brier"], label="CoxPH")
ax.set_xlabel("Time")
ax.set_ylabel("Oracle Brier score")
ax.set_title("Prediction error across the IBS interval")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## Median predictions versus oracle event times

In [ ]:
models = [
    ("Latent DVFM", "latent_predicted_median_event_time", "latent_median_crossed"),
    ("No-latent", "no_latent_predicted_median_event_time", "no_latent_median_crossed"),
    ("CoxPH", "coxph_predicted_median_event_time", "coxph_median_crossed"),
]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (title, prediction_col, crossing_col) in zip(axes, models):
    subset = predictions.loc[predictions[crossing_col].astype(bool)]
    ax.scatter(
        subset["true_event_time"],
        subset[prediction_col],
        alpha=0.35,
        s=18,
    )
    if len(subset):
        lo = min(subset["true_event_time"].min(), subset[prediction_col].min())
        hi = max(subset["true_event_time"].max(), subset[prediction_col].max())
        ax.plot([lo, hi], [lo, hi], linestyle="--")
    ax.set_xlabel("Oracle true event time")
    ax.set_ylabel("Predicted median event time")
    ax.set_title(f"{title} (n={len(subset)})")
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


## Subject-level median absolute errors

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, column in [
    ("Latent DVFM", "latent_median_absolute_error"),
    ("No-latent", "no_latent_median_absolute_error"),
    ("CoxPH", "coxph_median_absolute_error"),
]:
    values = predictions[column].replace([np.inf, -np.inf], np.nan).dropna()
    ax.hist(values, bins=45, density=True, alpha=0.4, label=f"{label} (n={len(values)})")
ax.set_xlabel("Median-time absolute error")
ax.set_ylabel("Density")
ax.set_title("Error distributions among valid median crossings")
ax.legend()
plt.show()


## Interpretation

Median-time MAE should always be read together with the crossing fraction. A model can appear to have a favorable MAE if it only produces finite medians for an easy subset. Oracle IBS remains the most complete primary metric because it evaluates the full survival curve for every subject over a fixed interval.
